## Today we convert your Colab project into a REAL deployable cloud project using only FREE open-source models (no API key).

Because Colab cannot stay live 24/7, we will use Colab to prepare the production project files, then you deploy them to Render + Streamlit Cloud.

Think of this session as building the deployment-ready production repo.

## FINAL GOAL

I will leave with a GitHub-ready project that can be deployed LIVE.

Architecture:

Frontend (Streamlit Cloud) → Backend API (Render) → HuggingFace Model → SQLite DB

All FREE.

## Step 0 — Open New Google Colab

Create new notebook:

Deployment_Project_Preparation.ipynb

We will generate all project files inside Colab.

## STEP 1 — Create Project Folder

Run:
```
import os
os.makedirs("AI_Assistant_Project", exist_ok=True)
```

In [2]:
import os
os.makedirs("AI_Assistant_Project", exist_ok=True)

### STEP 2 — Create DATABASE FILE (db.py)

Run this cell to generate the file automatically:

In [3]:
db_code = '''
import sqlite3

def init_db():
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chats (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        message TEXT,
        response TEXT
    )
    """)
    conn.commit()
    conn.close()

def save_chat(user_id, message, response):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO chats (user_id, message, response) VALUES (?, ?, ?)",
        (user_id, message, response)
    )
    conn.commit()
    conn.close()

def get_history(user_id):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT message, response FROM chats WHERE user_id=?",
        (user_id,)
    )
    rows = cursor.fetchall()
    conn.close()
    return rows
'''

open("AI_Assistant_Project/db.py","w").write(db_code)
print("db.py created")

db.py created


## STEP 3 — Create BACKEND API (api.py)

This will run on Render cloud.

In [4]:
api_code = '''
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import pipeline
from db import init_db, save_chat, get_history

app = Flask(__name__)
CORS(app)

# Load FREE model (no API key)
chatbot = pipeline("text-generation", model="distilgpt2")

init_db()

roles = {
    "teacher": "You are a helpful AI teacher.",
    "interview": "You are a job interview coach.",
    "strict": "Answer in 2 short lines."
}

@app.route("/chat", methods=["POST"])
def chat():
    data = request.json
    user_id = data["user_id"]
    role = data["role"]
    message = data["message"]

    history = get_history(user_id)

    prompt = roles[role] + "\\n"
    for msg, res in history:
        prompt += f"User: {msg}\\nBot: {res}\\n"
    prompt += f"User: {message}\\nBot:"

    output = chatbot(prompt, max_length=200, do_sample=True)
    reply = output[0]["generated_text"].split("Bot:")[-1]

    save_chat(user_id, message, reply)

    return jsonify({"response": reply})

app.run(host="0.0.0.0", port=10000)
'''

open("AI_Assistant_Project/api.py","w").write(api_code)
print("api.py created")

api.py created


## STEP 4 — Create FRONTEND (Streamlit app.py)

In [5]:
app_code = '''
import streamlit as st
import requests
import uuid

st.title("AI Assistant")

API_URL = "PASTE_RENDER_URL_HERE/chat"

if "user_id" not in st.session_state:
    st.session_state.user_id = str(uuid.uuid4())

role = st.selectbox("Select Role", ["teacher","interview","strict"])
message = st.text_input("Ask something")

if st.button("Send"):
    res = requests.post(API_URL, json={
        "user_id": st.session_state.user_id,
        "role": role,
        "message": message
    })
    st.write(res.json()["response"])
'''

open("AI_Assistant_Project/app.py","w").write(app_code)
print("app.py created")

app.py created


## STEP 5 — Create requirements.txt

In [6]:
req = '''
flask
flask-cors
transformers
torch
streamlit
requests
'''

open("AI_Assistant_Project/requirements.txt","w").write(req)

56

## STEP 6 — Create README.md

In [7]:
readme = '''
# AI Assistant (Open Source)

Multi-user AI chatbot using HuggingFace + Flask + Streamlit.

Features:
- No API key required
- Persistent chat database
- Multi-user sessions
- Cloud deployable
'''

open("AI_Assistant_Project/README.md","w").write(readme)

193

## STEP 7 — Zip Project (Download)
```
import shutil
shutil.make_archive("AI_Assistant_Project", 'zip', "AI_Assistant_Project")
```
Download the zip from Colab files panel.

In [8]:
import shutil
shutil.make_archive("AI_Assistant_Project", 'zip', "AI_Assistant_Project")

'/content/AI_Assistant_Project.zip'

# NOW REAL DEPLOYMENT (Outside Colab)

## STEP 8 — Upload to GitHub

- Create new repo: AI-Assistant
- Upload extracted files.

## STEP 9 — Deploy Backend on Render

Go → https://render.com

Create Web Service

Settings:

Build command:
```
pip install -r requirements.txt
```

Start command:
```
python api.py
```
After deploy you get URL:

👉 https://ai-assistant.onrender.com

## STEP 10 — Update Frontend URL

Open app.py in GitHub and replace:
```
API_URL = "PASTE_RENDER_URL_HERE/chat"
```
Example:
```
API_URL = "https://ai-assistant.onrender.com/chat"
```
Commit changes.

## STEP 11 — Deploy Frontend on Streamlit Cloud

Go → https://streamlit.io/cloud

Deploy repo → choose app.py

You will get:

👉 https://ai-assistant.streamlit.app

### YOU ARE LIVE

You now have:

| Component   | Status               |
| ----------- | -------------------- |
| Backend API | Live on Render       |
| Frontend UI | Live on Streamlit    |
| Model       | HuggingFace FREE     |
| Database    | SQLite persistent    |
| Users       | Multi-user supported |


### This is Portfolio-Level Project

"I deployed a multi-user AI assistant using open-source LLMs on cloud."